# Spin8: HF → A100 → GGUF
**2026-11-23修正版。Google Colabの「最新（推奨）」GPUランタイム用。**

「ランタイム → ランタイムのタイプを変更」で **A100 GPU・ハイメモリ・最新（推奨）** を選び、上から実行します。
実際のGPU名も確認し、L4などが割り当てられた場合はビルド・ダウンロード・量子化を始める前に停止します。
`Qwen/Qwen3.8-27B`をHFから自動取得します。追加ZIPや元GGUFの手動準備は不要です。
先に小型モデルでCUDAカーネル・推論・方向射影・APIを検証し、その後27Bを変換します。

通常のSpin8量子化は初期設定のままで進みます。数学差分を使う場合は`MATH_DELTA=True`。
対応する祖先版と数学ドナーを自動取得します。新たな回答データによる学習は行いません。
方向除去を使う場合だけ`DIRECTION_PATH`に検証済みの.npyを指定してください。
Drive上の方向ファイルを指定すると、最初のセルでDriveを接続します。
`OUTPUT_NAME`が空なら、モデル名と設定・方向ファイルに応じた名前を自動生成します。
数学差分の強さや方向を変更した実験は、完成品を別のファイルとして保存します。
重みから「拒否」の意味を自動判定したり、架空の方向ベクトルを生成したりはしません。

**このGGUFは同梱のSpin8対応ランタイムが必要です。** 通常のllama.cpp・Ollama・LM Studioでの互換性はありません。
2.125 bit/weightは対象行列の符号化ペイロードです。非量子化テンソル・回転行列・メタデータは別に必要です。
数学能力向上・拒否率低下はこの動作テストでは判定できません。
後半ではGSM8Kの固定64問で数学の正答率も測ります。初期設定は元F16との比較です。
参照F16のための追加容量が足りない場合は候補モデルだけを測定し、比較未実施と記録します。

公開HFモデルの取得にトークンは不要です。HFへのアップロードは最後の任意セルで行います。
VMが存続していれば同じ作業フォルダーでダウンロード・量子化を再開できます。
VMの削除後はローカルキャッシュも消えるため、完成後にDriveまたはHFへ保存してください。


In [ ]:
# @title 1. Configuration and embedded source preparation
from pathlib import Path
import base64, hashlib, io, json, os, re, shutil, signal, subprocess, sys, tempfile, zipfile

WORK_DIR = "/content/spin8-work" # @param {type:"string"}
SOURCE_REPO = "Qwen/Qwen3.8-27B" # @param {type:"string"}
SOURCE_REVISION = "main" # @param {type:"string"}
MATH_DELTA = False # @param {type:"boolean"}
MATH_ALPHA = 0.25 # @param {type:"number"}
DIRECTION_PATH = "" # @param {type:"string"}
OUTPUT_NAME = "" # @param {type:"string"}
HF_REPO = "" # @param {type:"string"}
HF_PRIVATE = True # @param {type:"boolean"}

def require_a100():
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is unavailable. Select an A100 GPU runtime before running this notebook.")
    _gpu = torch.cuda.get_device_properties(0)
    if "A100" not in _gpu.name:
        raise RuntimeError("An A100 is required; Colab allocated " + _gpu.name + ". Select an A100 runtime before download or quantisation.")
    return {"gpu": _gpu.name, "vram_gib": round(_gpu.total_memory / 2**30, 2),
            "torch": str(torch.__version__), "cuda": torch.version.cuda}
print("A100 runtime:", json.dumps(require_a100()), flush=True)

# The source bundle is embedded; no ZIP upload is required.
_payload_sha = "b7515ce2d5dc6be68e52a7e607b2722d0e194175f3b8775c584b0ccaf43358e3"
_payload_blob = base64.b64decode("...", validate=True)
assert hashlib.sha256(_payload_blob).hexdigest() == _payload_sha, "Embedded payload checksum mismatch"
PAYLOAD = Path("/content") / ("spin8-payload-" + _payload_sha[:12])
if not PAYLOAD.exists():
    _temporary = Path(tempfile.mkdtemp(prefix="spin8-unpack-", dir="/content"))
    try:
        with zipfile.ZipFile(io.BytesIO(_payload_blob)) as _archive:
            for _item in _archive.infolist():
                _relative = Path(_item.filename)
                assert not _relative.is_absolute() and ".." not in _relative.parts
                assert not ((_item.external_attr >> 16) & 0o170000 == 0o120000)
                _archive.extract(_item, _temporary)
        os.rename(_temporary, PAYLOAD)
    finally:
        if _temporary.exists():
            shutil.rmtree(_temporary)
_manifest = json.loads((PAYLOAD / "MANIFEST.json").read_text())
for _item in _manifest["files"]:
    _path = PAYLOAD / _item["path"]
    assert _path.is_file() and not _path.is_symlink(), _item["path"]
    assert _path.stat().st_size == _item["bytes"], _item["path"]
    assert hashlib.sha256(_path.read_bytes()).hexdigest() == _item["sha256"], _item["path"]
del _payload_blob
WORK = Path(WORK_DIR).expanduser().resolve()
assert str(WORK).startswith("/content/"), "Use a local /content working directory"
assert not WORK.is_relative_to(Path("/content/drive")), "Build/download in local /content; export final files to Drive below"
WORK.mkdir(parents=True, exist_ok=True)
def file_sha256(path):
    _digest = hashlib.sha256()
    with Path(path).open("rb") as _stream:
        for _block in iter(lambda: _stream.read(8 << 20), b""):
            _digest.update(_block)
    return _digest.hexdigest()
_direction_hash = None
if DIRECTION_PATH.strip():
    _direction = Path(os.path.abspath(str(Path(DIRECTION_PATH).expanduser())))
    if _direction.is_relative_to(Path("/content/drive")):
        from google.colab import drive
        drive.mount("/content/drive")
    assert _direction.is_file(), "DIRECTION_PATH must name an existing .npy file"
    DIRECTION_PATH = str(_direction.resolve())
    _direction_hash = file_sha256(DIRECTION_PATH)
else:
    DIRECTION_PATH = ""
CONFIG = WORK / "config.json"
_suffix = ("-MathDelta" if MATH_DELTA else "") + ("-Projected" if DIRECTION_PATH else "")
_recipe = {"source_repo": SOURCE_REPO, "source_revision": SOURCE_REVISION,
           "math_delta": MATH_DELTA, "math_alpha": MATH_ALPHA if MATH_DELTA else None,
           "direction_sha256": _direction_hash, "payload_sha256": _payload_sha,
           "seed": 81237, "scale_steps": 17, "chunk_groups": 8192, "tile_elements": 1048576}
_recipe_hash = hashlib.sha256(json.dumps(_recipe, sort_keys=True, allow_nan=False).encode()).hexdigest()[:10]
_model_name = re.sub(r"[^A-Za-z0-9._-]+", "-", SOURCE_REPO.rsplit("/", 1)[-1]).strip("._-") or "model"
_output_name = OUTPUT_NAME.strip() or (_model_name + _suffix + "-Spin8-" + _recipe_hash + ".gguf")
assert re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._-]*\.gguf", _output_name), "OUTPUT_NAME must be a simple filename ending in .gguf"
_config = {"work_dir": str(WORK), "payload_dir": str(PAYLOAD), "source_repo": SOURCE_REPO,
           "source_revision": SOURCE_REVISION, "device": "cuda", "math_delta": MATH_DELTA,
           "math_alpha": MATH_ALPHA, "direction_path": DIRECTION_PATH,
           "output_name": _output_name,
           "seed": 81237, "scale_steps": 17, "chunk_groups": 8192,
           "tile_elements": 1048576, "hf_repo": HF_REPO, "hf_private": HF_PRIVATE}
CONFIG.write_text(json.dumps(_config, indent=2) + "\n")
def run_spin8(*arguments):
    require_a100()
    _command = [sys.executable, "-u", str(PAYLOAD / "colab_app" / arguments[0]),
                *map(str, arguments[1:])]
    _process = subprocess.Popen(_command, cwd=WORK, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1,
                                start_new_session=True)
    try:
        for _line in _process.stdout:
            print(_line, end="", flush=True)
        _returncode = _process.wait()
        if _returncode:
            raise subprocess.CalledProcessError(_returncode, _command)
    except BaseException:
        # Stop this stage's process group, including compiler/server children.
        try:
            os.killpg(_process.pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        try:
            _process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            try:
                os.killpg(_process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            _process.wait()
        raise
    finally:
        _process.stdout.close()
def verified_artifacts(include_benchmarks=False):
    _gate_code = (
        "import contextlib, json, sys\n"
        "from pathlib import Path\n"
        "sys.path.insert(0, sys.argv[1])\n"
        "from colab_app import pipeline, bootstrap\n"
        "config = pipeline.load_config(sys.argv[2])\n"
        "with pipeline.pipeline_lock(config), contextlib.redirect_stdout(sys.stderr):\n"
        "    build, conversion, verification = pipeline.publication_gate(config)\n"
        "    locations = pipeline.paths(config)\n"
        "    files = [conversion['output'], bootstrap.file_info(locations['conversion']), bootstrap.file_info(locations['verification'])]\n"
        "    files = [{**item, 'relative_path': Path(item['path']).name} for item in files]\n"
        "    if sys.argv[3] == '1':\n"
        "        for relative, path in pipeline.matching_benchmark_files(config, conversion, verification, build):\n"
        "            files.append({**bootstrap.file_info(path), 'relative_path': 'benchmarks/' + locations['output'].stem + '/' + Path(relative).name})\n"
        "    result = {'output': conversion['output'], 'server': build['llama-server'], 'files': files}\n"
        "print(json.dumps(result))\n"
    )
    _gate = subprocess.run([sys.executable, "-c", _gate_code, str(PAYLOAD), str(CONFIG),
                           "1" if include_benchmarks else "0"],
                          cwd=WORK, check=True, text=True, stdout=subprocess.PIPE)
    return json.loads(_gate.stdout)
print("Ready:", SOURCE_REPO, "| work:", WORK, "| output:", _output_name, "| embedded source verified")


## 2. 環境を保ち、Spin8をビルドして実機検証

Colab既存のPyTorch/CUDA/NumPyを維持します。実際のGPUアーキテクチャを検出し、
CUDA計算と推論の配置まで検証します。段階別の検証ログは`WORK_DIR/reports/`に残り、
標準出力・標準エラーもノートブックへ逐次表示します。中断時は実行したプロセス群を終了します。

CPU参照との数値比較ではKVキャッシュをFP32に揃え、CUDA側の比較子プロセスだけで
`GGML_CUDA_SPIN8_VALIDATE_F32=1`と`NVIDIA_TF32_OVERRIDE=0`を使います。
通常の行列積に混ざるTF32の影響を除くための設定で、Spin8カーネルは変更しません。
許容誤差は従来の`atol=rtol=1e-4`です。生成APIの確認にはこの比較専用設定を渡しません。
エラー時は失敗した段階が表示されます。環境が壊れている場合は新しいランタイムで再開してください。


In [ ]:
# @title 2. Build and mandatory CUDA runtime checks
run_spin8("bootstrap.py", "prepare", "--work-dir", WORK,
          "--payload-dir", PAYLOAD, "--device", "cuda", "--jobs", "4")
_build = json.loads((WORK / "reports/build-report.json").read_text())
assert _build["status"] == "PASS" and _build["cuda_device_execution"]
print("Spin8 CUDA runtime checks: PASS")


## 3. HFから取得

完全なリポジトリIDとコミットを固定し、全シャードをサイズ・ハッシュ・Safetensors構造で検証します。
最新版は最初の取得時に固定され、途中の再実行で別版へ変わりません。
数学差分の場合はドナーが宣言した祖先版を使います。
空き容量を確認してから大きなファイルの取得を開始します。


In [ ]:
# @title 3. Plan storage and download verified HF weights
run_spin8("pipeline.py", "--config", CONFIG, "plan")
run_spin8("pipeline.py", "--config", CONFIG, "download")


## 4. Spin8 GGUFへ直接変換

公式HF変換処理のテンソル並べ替え・融合・トークナイザー処理を使います。
中間の巨大な非量子化GGUFは作りません。量子化済みのテンソルをチェックサム付きで再利用します。
画像エンコーダーとMTPを除くテキスト推論モデルを出力します。
後のベンチマークで元F16との比較を有効にした場合だけ、別途参照GGUFを作ります。

既定は`chunk_groups=8192`、`tile_elements=1048576`、`scale_steps=17`、`seed=81237`です。
FP32で候補を比較し、完了したテンソルから再開します。変換の最初には元シャードのハッシュを再確認します。


In [ ]:
# @title 4. Convert HF weights directly to Spin8 GGUF
run_spin8("pipeline.py", "--config", CONFIG, "convert")


## 5. 完成した27Bで推論確認

実際の出力ファイルを読み込み、有限なlogits・Spin8演算のGPU配置・生成APIを確認します。
合格は実行経路の確認であり、数学ベンチマークでの改善を意味しません。


In [ ]:
# @title 5. Verify the actual exported model
run_spin8("pipeline.py", "--config", CONFIG, "verify")
print("Output directory:", WORK / "outputs")


## 6. 数学ベンチマーク: GSM8Kで元F16と比較

初期設定では、固定した64問を元F16とSpin8の両方に解かせ、正答率・問題ごとの結果・生成速度を記録します。
同じチャットテンプレートと入力トークンで比較します。数学差分や方向除去を使った場合も、
参照は編集前の同じ祖先モデルです。量子化だけの影響を分けて調べる場合は、編集なしの設定も実行してください。

`BUILD_F16_BASELINE=True`は参照用F16を自動作成します。27Bでは追加で約54 GBが必要です。
完成した参照はハッシュを検証して再利用します。空き容量が不足する場合は候補だけを測り、改善比較は行いません。
既存の参照GGUFを使う場合は`BASELINE_GGUF`に指定できます。
40 GB GPUでは参照の一部がCPUで動く場合があり、速度の差には配置の差も含まれます。

64問は動作と傾向を見るための小規模評価です。全問題を評価するときは`BENCHMARK_FULL=True`にします。
`THINKING`と生成上限は両モデルで共通です。データセットの答えは採点に使い、重み学習には使いません。
問題数・思考設定・生成上限を変えると別の実行フォルダーを作り、同じ条件では完了済みの問題から再開します。
不足するParquet読込用のPyArrowだけを追加できます。既存のPyTorch・NumPyは変更しません。


In [ ]:
# @title 6. Benchmark mathematics against the original F16 model
RUN_BENCHMARK = True # @param {type:"boolean"}
BUILD_F16_BASELINE = True # @param {type:"boolean"}
BASELINE_GGUF = "" # @param {type:"string"}
BENCHMARK_FULL = False # @param {type:"boolean"}
BENCHMARK_SAMPLES = 64 # @param {type:"integer"}
BENCHMARK_SEED = 20261121 # @param {type:"integer"}
MAX_GENERATED_TOKENS = 2048 # @param {type:"integer"}
THINKING = "off" # @param ["off", "on"]
INSTALL_MISSING_PYARROW = True # @param {type:"boolean"}
if RUN_BENCHMARK:
    _verified = verified_artifacts()
    _candidate = Path(_verified["output"]["path"])
    _benchmark_root = WORK / "reports" / "benchmarks" / _candidate.stem
    _benchmark_root.mkdir(parents=True, exist_ok=True)
    _baseline = None
    _baseline_hash = None
    if BASELINE_GGUF.strip():
        _baseline = Path(os.path.abspath(str(Path(BASELINE_GGUF).expanduser())))
        if _baseline.is_relative_to(Path("/content/drive")):
            from google.colab import drive
            drive.mount("/content/drive")
        assert _baseline.is_file(), "BASELINE_GGUF must name an existing reference model"
        _baseline = _baseline.resolve()
        _baseline_hash = file_sha256(_baseline)
    elif BUILD_F16_BASELINE:
        _baseline_report = _benchmark_root / "baseline-preparation.json"
        run_spin8("baseline.py", "--config", CONFIG, "--report", _baseline_report,
                  "--skip-if-insufficient-space")
        _prepared = json.loads(_baseline_report.read_text())
        if _prepared["status"] == "COMPLETE":
            _baseline = Path(_prepared["output"]["path"])
            _baseline_hash = _prepared["output"]["sha256"]
        else:
            assert _prepared["status"] == "SKIPPED_SPACE", "F16 reference preparation did not complete"
            print("F16参照の追加容量が足りないため候補だけを測ります。元モデルとの改善比較は未実施です。")
    _benchmark_recipe = {"candidate": _verified["output"], "server": _verified["server"],
                         "baseline_path": str(_baseline) if _baseline else None,
                         "baseline_sha256": _baseline_hash, "full": BENCHMARK_FULL,
                         "samples": None if BENCHMARK_FULL else BENCHMARK_SAMPLES,
                         "seed": BENCHMARK_SEED, "max_tokens": MAX_GENERATED_TOKENS,
                         "thinking": THINKING, "ctx_size": 4096,
                         "benchmark_code_sha256": file_sha256(PAYLOAD / "colab_app/benchmark.py")}
    _benchmark_id = hashlib.sha256(json.dumps(_benchmark_recipe, sort_keys=True, allow_nan=False).encode()).hexdigest()[:16]
    _benchmark_dir = _benchmark_root / "runs" / _benchmark_id
    _arguments = ["benchmark.py", "--server-bin", _verified["server"]["path"],
                  "--candidate-gguf", _candidate, "--output-dir", _benchmark_dir,
                  "--max-tokens", str(MAX_GENERATED_TOKENS), "--thinking", THINKING,
                  "--seed", str(BENCHMARK_SEED), "--ctx-size", "4096",
                  "--gpu-layers", "auto", "--baseline-gpu-layers", "auto"]
    _arguments += ["--full"] if BENCHMARK_FULL else ["--samples", str(BENCHMARK_SAMPLES)]
    if _baseline is not None:
        _arguments += ["--baseline-gguf", _baseline]
    if INSTALL_MISSING_PYARROW:
        _arguments += ["--install-missing"]
    run_spin8(*_arguments)
    _result = json.loads((_benchmark_dir / "benchmark.json").read_text())
    assert _result["status"] == "COMPLETE", "Benchmark did not complete"
    assert _result["inputs"]["candidate"]["sha256"] == _verified["output"]["sha256"], "Benchmark candidate differs"
    assert _result["inputs"]["server"]["sha256"] == _verified["server"]["sha256"], "Benchmark runtime differs"
    assert file_sha256(_benchmark_dir / "benchmark.md") == _result["artifacts"]["benchmark_md_sha256"], "Benchmark report checksum differs"
    # Keep every run intact; publish the latest complete report pair at
    # the stable location used by the Drive/HF artifact gates.
    for _name in ("benchmark.md", "benchmark.json"):
        _fd, _pending_name = tempfile.mkstemp(prefix="." + _name + ".", dir=_benchmark_root)
        os.close(_fd)
        _pending = Path(_pending_name)
        try:
            shutil.copyfile(_benchmark_dir / _name, _pending)
            os.replace(_pending, _benchmark_root / _name)
        finally:
            _pending.unlink(missing_ok=True)
    from IPython.display import Markdown, display
    display(Markdown((_benchmark_dir / "benchmark.md").read_text()))
    print("Benchmark results:", _benchmark_dir)
else:
    print("Benchmark disabled; no accuracy comparison was performed in this run")


## 任意: Google Driveへ完成品を保存

下の`SAVE_TO_DRIVE`を有効にすると、現在の設定と検証済みファイルの一致を再確認し、
そのGGUFと変換・検証レポートの3ファイルを専用フォルダーへ保存します。
現在のモデルと実行環境に一致する完了済みベンチマークがあれば、その結果も保存します。
大きな参照F16はコピーしません。
同名で内容の異なる既存ファイルは上書きしません。


In [ ]:
# @title Optional: save completed files to Drive
SAVE_TO_DRIVE = False # @param {type:"boolean"}
DRIVE_DIRECTORY = "/content/drive/MyDrive/Spin8_Qwen38_20261123" # @param {type:"string"}
if SAVE_TO_DRIVE:
    _verified_files = verified_artifacts(include_benchmarks=True)["files"]
    assert len(_verified_files) >= 3, "Unexpected verified artifact inventory"
    from google.colab import drive
    drive.mount("/content/drive")
    _destination = Path(DRIVE_DIRECTORY)
    _destination.mkdir(parents=True, exist_ok=True)
    for _artifact in _verified_files:
        _source = Path(_artifact["path"])
        _relative = Path(_artifact["relative_path"])
        assert not _relative.is_absolute() and ".." not in _relative.parts, "Invalid artifact destination"
        _target = _destination / _relative
        _target.parent.mkdir(parents=True, exist_ok=True)
        _hash = _artifact["sha256"]
        assert _source.is_file() and _source.stat().st_size == _artifact["bytes"], "Verified source changed: " + _source.name
        if _target.exists():
            assert _target.is_file() and file_sha256(_target) == _hash, "Different existing Drive file: " + _target.name
            continue
        _fd, _pending_name = tempfile.mkstemp(prefix="." + _source.name + ".", suffix=".partial", dir=_target.parent)
        os.close(_fd)
        _pending = Path(_pending_name)
        try:
            shutil.copyfile(_source, _pending)
            assert file_sha256(_pending) == _hash, "Drive copy checksum mismatch"
            if _target.exists():
                assert _target.is_file() and file_sha256(_target) == _hash, "Different existing Drive file: " + _target.name
            else:
                os.replace(_pending, _target)
        finally:
            _pending.unlink(missing_ok=True)
        print("Saved:", _target.name)
    print("Copied and read back through DriveFS:", _destination)
    print("This verifies the mounted copy, not independent remote synchronization.")
else:
    print("Drive save disabled; completed files remain in", WORK / "outputs")


## 任意: Hugging Faceへアップロード

セル1の`HF_REPO`に自分の`ユーザー名/リポジトリ名`を設定し、Colabシークレットの`HF_TOKEN`に書き込み権限のあるトークンを入れます。
下の`UPLOAD_TO_HF`を有効にして実行します。トークンはノートブックや設定ファイルに書き込みません。
初期設定は非公開です。完成GGUF・由来・動作確認・専用ランタイムのソースを一緒にアップロードします。


In [ ]:
# @title Optional: publish the verified result to HF
UPLOAD_TO_HF = False # @param {type:"boolean"}
if UPLOAD_TO_HF:
    assert HF_REPO.strip(), "Set HF_REPO in cell 1 and rerun cell 1"
    from google.colab import userdata
    _token = userdata.get("HF_TOKEN")
    assert _token, "Add HF_TOKEN to Colab Secrets"
    _upload_env = os.environ.copy()
    _upload_env["HF_TOKEN"] = _token
    try:
        subprocess.run([sys.executable, "-u", str(PAYLOAD / "colab_app/pipeline.py"),
                        "--config", str(CONFIG), "publish"], cwd=WORK, env=_upload_env, check=True)
    finally:
        _upload_env.pop("HF_TOKEN", None)
        del _token, _upload_env
else:
    print("HF upload disabled; set HF_REPO and enable this cell when ready")
